# vgc-bot — end to end on a GPU box

Works on any Jupyter host (remote A100/L4 box, Colab Pro, ...). Run from the
repo root with the repo already cloned; every section works independently
given the previous section's artifacts in `artifacts/`.

Pipeline: **setup → data → train → evaluate → env benchmark → search**.

## 0. Setup — Python deps, Node + pinned sim (no root needed)

In [ ]:
# On Colab only, clone first:
# %cd /content
# !git clone https://github.com/YOURNAME/vgc-bot.git
# %cd /content/vgc-bot
!pip install -q -r requirements.txt

In [ ]:
# Node 22: use the system install if present, otherwise download the official
# standalone tarball into artifacts/ (no apt, no root — works on any Linux).
# The kernel PATH is extended, so every later !cell and every sidecar spawned
# by the python scripts finds it. For a plain terminal later:
#   export PATH=$PWD/artifacts/node-dist/bin:$PATH
import os, platform, shutil, subprocess, tarfile, urllib.request
from pathlib import Path

node_dir = Path("artifacts/node-dist").resolve()
if not shutil.which("node"):
    if not (node_dir / "bin" / "node").exists():
        arch = {"x86_64": "x64", "AMD64": "x64", "aarch64": "arm64",
                "arm64": "arm64"}[platform.machine()]
        name = f"node-v22.12.0-linux-{arch}"
        url = f"https://nodejs.org/dist/v22.12.0/{name}.tar.xz"
        print("downloading", url)
        Path("artifacts").mkdir(exist_ok=True)
        urllib.request.urlretrieve(url, "artifacts/node.tar.xz")
        with tarfile.open("artifacts/node.tar.xz") as tf:
            tf.extractall("artifacts", filter="tar")   # keep exec bits/symlinks
        (Path("artifacts") / name).rename(node_dir)
    os.environ["PATH"] = str(node_dir / "bin") + os.pathsep + os.environ["PATH"]
print(subprocess.run(["node", "--version"], capture_output=True, text=True).stdout)

In [ ]:
# Pinned pokemon-showdown (git commit with the Champions formats) + @smogon/calc.
# The git install ships no dist/, so build it in place (~2 min).
!mkdir -p artifacts/node
%cd artifacts/node
![ -f package.json ] || echo '{"name":"vgc-bot-node","private":true}' > package.json
!npm install --no-audit --no-fund github:smogon/pokemon-showdown#e440c4a18385274f10c405d0b158b6a962ce6d94 @smogon/calc@0.11.0
!cd node_modules/pokemon-showdown && (npm install --no-audit --no-fund || true) && node build && ls dist/sim/index.js
%cd ../..

In [ ]:
# Checkpoints default to artifacts/checkpoints (config.py). On Colab, put them
# on Drive so training survives the session; anywhere else this is a no-op.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    get_ipython().system("mkdir -p /content/drive/MyDrive/vgc-bot/checkpoints")
    get_ipython().system("sed -i 's|Path(\"artifacts/checkpoints\")|Path(\"/content/drive/MyDrive/vgc-bot/checkpoints\")|' config.py")
except ImportError:
    print("not Colab: checkpoints stay in artifacts/checkpoints")
!mkdir -p artifacts/checkpoints
!grep checkpoint_dir config.py

## 1. Data — download, parse, prep (beliefs + damage features + tokenize)

In [ ]:
!python env.py --dump-dex   # champions dex for stat/priority math (needs node)
from config import CFG
import data
if [f for f in CFG.dataset_files if not (CFG.data_dir / f).exists()]:
    data.download()             # ~630 MB from Hugging Face
else:
    print("dataset already present, skipping download")
!python data.py parse

In [ ]:
# The slow step (particle filter + damage calc per transition).
!python data.py prep

## 2. Train — behavior cloning (resumes from ckpt_last.pt automatically)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir artifacts/checkpoints/tb
!python train.py

## 3. Evaluate — predictor benchmarks on held-out battles

In [ ]:
!python evaluate.py
# model debugging (see README 'Debug modes'):
# !python evaluate.py --worst 20   # decode the positions it gets most wrong
# !python evaluate.py --aux        # aux set-prediction head vs oracle sheets

## 4. Env benchmark — sidecar save/restore proof + steps/sec (sets the search budget)

In [ ]:
!python env.py --benchmark 3000

## 5. Phase 2 — search
Reconstruction selftest first (proves the sim pin), then the belief-filter audit,
the scenario assertions (incl. the Metagross/Kingambit mixed-strategy gate) and a
watchable search-vs-search game.

NOTE: if the prepped shards/checkpoints predate the phase-2 tokenizer layout
(belief tokens + damage ranges, 537 tokens), re-run sections 1-2 first.

In [ ]:
!python env.py --selftest      # mid-battle reconstruction proof
!python beliefs.py --audit 300 # particle-filter breakdown on held-out games

In [ ]:
!python scenarios.py           # endgame assertions; works pre-training too
!python scenarios.py --mine    # real-replay endgame candidates -> artifacts/
# profiling: !python scenarios.py --debug   /   --cprofile search.prof

In [ ]:
!python observe_game.py --games 1 --temp 0.5   # needs ckpt_best.pt

In [ ]:
!python teams.py --validate    # replica teams through the sim's validator
# then swap in real sheets for anything flagged:  !python teams.py --mine

## Phase 3 (next)
Bot-vs-bot round-robin Elo: random / max-damage / policy-only / policy+search.
The policy-only vs policy+search gap is the evidence that search adds value.

(Human-vs-bot play — `python play.py` — needs a browser next to the server, so
run it on your own machine, not from this notebook.)